# Reading the SARB Between the Lines

Reproducible walkthrough of the pipeline behind the article of the same name: a replication and update of du Rand, Erasmus, Hollander, Reid & van Lill (2021), extended from their 1994–2020 window through July 2026.

This notebook does not reimplement anything — every step below just calls the corresponding script in `scripts/` and narrates what it does and why. See `README.md` for the full repo map and `Methodology & AI Usage` section.

**Runtime note:** Section 1 (scraping) hits the live SARB site and can take a while (and is not guaranteed to be re-runnable forever if the site changes). Everything from Section 2 onward runs in seconds against the datasets already checked into `data/processed/`. By default, Section 1's scraping cells are **not executed** — the notebook loads the existing datasets instead. Flip `RUN_SCRAPING = True` below if you want to re-scrape from scratch.

In [ ]:
import sys, json, csv
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "scripts"))
import config

RUN_SCRAPING = False  # set True to re-scrape from the live SARB site (slow)

import matplotlib.pyplot as plt
from IPython.display import Image, display
%matplotlib inline

## 1. Corpus collection

`fetch_statement_list.py` / `fetch_speech_list.py` paginate the SARB site's internal Solr search API to build a full index of every MPC statement (since October 1999) and every speech (since August 1994). `scrape_statements.py` / `scrape_speeches.py` then visit each detail page and extract the full text — the SARB site has used three different templates over 27 years (plain HTML, HTML shell + linked PDF, HTML with data widgets), and `scrape_common.py` detects and handles all three.

`scrape_statements.py` **overwrites the full statements dataset on every run** (not incremental); `scrape_speeches.py` is resumable (merges by URL).

In [ ]:
if RUN_SCRAPING:
    import fetch_statement_list, scrape_statements, fetch_speech_list, scrape_speeches
    fetch_statement_list.main()
    scrape_statements.main()
    fetch_speech_list.main()
    scrape_speeches.main()

statements = json.loads(config.DATASET_JSON.read_text(encoding="utf-8"))
speeches = json.loads(config.SPEECHES_DATASET_JSON.read_text(encoding="utf-8"))
print(f"{len(statements)} MPC statements on record")
print(f"{len(speeches)} speeches on record "
      f"({sum(1 for r in speeches if r.get('scrape_status')=='ok')} scraped successfully)")

## 2. Rate-decision cross-validation

Regex over the statement text (`parsers.py`) gives a first pass at each meeting's hike/cut/hold decision, but isn't 100% reliable on unusual phrasing. `build_rate_from_series.py` cross-checks every meeting against an official daily SARB Policy Rate series: the rate on the meeting date vs. the first day strictly after it (the new rate takes effect the day after the announcement). Where the two disagree, the official series wins; `policy_rate_final` / `rate_action_final` are the fields used everywhere downstream.

In [ ]:
if RUN_SCRAPING:
    import build_rate_from_series
    build_rate_from_series.main()

from collections import Counter
actions = Counter(r.get("rate_action_final") for r in statements if r.get("rate_action_final"))
print("Rate decisions across all meetings with a known outcome:", dict(actions))

## 3. Language filtering

The SARB publishes official translations of some speeches (isiXhosa, isiZulu, Xitsonga) as separate entries in its speeches index — the same underlying communication event, counted twice, and (worse) silently returning a fake "neutral" sentiment score of 0.0 since the English-only dictionaries below don't match any words in them. `detect_language.py` scores each document by the share of common English stopwords among its alphabetic tokens; `flag_non_english.py` writes the resulting `is_english` field into both datasets. Every downstream step filters on `is_english is not False`.

In [ ]:
if RUN_SCRAPING:
    import detect_language, flag_non_english
    flag_non_english.main()

non_english = sum(1 for r in speeches if r.get("is_english") is False)
usable = sum(1 for r in speeches if r.get("scrape_status") == "ok" and r.get("is_english") is not False)
print(f"{non_english} non-English translations excluded; {usable} speeches usable for analysis")

## 4. Sentiment scoring — Henry (2008)

The primary sentiment method, replicating Erasmus & Hollander (2020): every word in a document is checked against Henry's (2008) hawkish/dovish word lists (`lexicon_henry.py`, transcribed from the original paper), and scored as

```
index = 2 × (hawkish_count − dovish_count) / (hawkish_count + dovish_count)
```

bounded between −2 and +2. `score_henry.py` runs this over statements; `score_speeches.py` runs the same function over speeches.

In [ ]:
if RUN_SCRAPING:
    import score_henry, score_speeches
    score_henry.main()
    score_speeches.run_henry(speeches)

def read_csv(path):
    with open(path, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

statement_scores = read_csv(config.PROCESSED_DIR / "scores_henry.csv")
speech_scores = read_csv(config.SPEECHES_SCORES_HENRY_CSV)

def mean(xs):
    xs = list(xs)
    return sum(xs) / len(xs)

s_vals = [float(r["henry_index"]) for r in statement_scores]
p_vals = [float(r["henry_index"]) for r in speech_scores]
print(f"Statements:  n={len(s_vals)}  mean={mean(s_vals):.3f}")
print(f"Speeches:    n={len(p_vals)}  mean={mean(p_vals):.3f}")
print("Speeches run systematically hotter -- see the article's Appendix for why.")

## 5. Combined index (per MPC meeting)

Following the paper's own logic that speeches between meetings are part of the same communication stream as the statement that follows them, `build_combined_index.py` assigns every speech to the window `(previous meeting, this meeting]` and computes an *exact* decomposition per meeting:

```
combined_index(M) = contrib_statement(M) + contrib_speech(M)
```

not a simple average — the two contributions are weighted by how many speeches fell in that window.

In [ ]:
if RUN_SCRAPING:
    import build_combined_index
    build_combined_index.main()

combined = read_csv(config.PROCESSED_DIR / "combined_index_by_meeting.csv")
action_by_date = {r["meeting_date"]: r.get("rate_action_final") for r in statements}

buckets = {"hike": [], "hold": [], "cut": []}
for r in combined:
    a = action_by_date.get(r["meeting_date"])
    if a in buckets:
        buckets[a].append(float(r["combined_index"]))

print("Sanity check -- does the combined index move with actual policy?")
for a, vals in buckets.items():
    print(f"  {a:5s}  n={len(vals):3d}  mean={mean(vals):.3f}")

## 6. Topic modelling (speeches only, LDA, k=12)

`topic_modeling_lda.py` runs `scikit-learn`'s `LatentDirichletAllocation` over the speeches corpus (unigrams + bigrams, `min_df=40`, `k=12`, `seed=99`). The choice of k=12 — more granular than the paper's k=6, less than an over-fitted k=15 — came from testing several configurations across multiple random seeds and checking which thematic splits were stable; see the module's docstring for the full calibration history, including the dead ends. Statements were also tested (as the paper does) and, like the paper, don't separate into distinguishable topics — this step is speeches-only for that reason.

In [ ]:
import topic_modeling_lda
topic_modeling_lda.main()  # deterministic given the fixed seed; fast (a few seconds)

## 7. Final charts

Two chart pairs, each replicating a figure from the original paper:

- `build_topic_words_bars.py` / `build_topic_evolution_bars.py` — the paper's Figure 6 (top terms per topic) and Figure 7 (topic weight by year), for our 6 final themes.
- `build_final_charts.py` / `build_final_contribution_charts.py` — interactive point-cloud + LOWESS + band charts (statements / speeches / combined) and the per-meeting contribution decomposition, in the paper's visual style. These are standalone HTML files in `reports/` — open them directly in a browser.

In [ ]:
import build_topic_words_bars, build_topic_evolution_bars
build_topic_words_bars.main()
build_topic_evolution_bars.main()

display(Image(filename=str(Path("charts/topic-words-bars.png"))))
display(Image(filename=str(Path("charts/topic-evolution-bars.png"))))

In [ ]:
import build_final_charts, build_final_contribution_charts
build_final_charts.main()
build_final_contribution_charts.main()
print("Wrote 8 interactive HTML charts to reports/ -- open them in a browser to view.")

## 8. The July 2026 check

This is the question that motivated pulling the whole pipeline back out: the SARB held at 7.00% on 23 July 2026 against a market majority pricing a hike. Is that a surprise by the Bank's own communication history? We pull every meeting whose combined index sits within 0.05 of July 2026's reading (+0.78) and look at what the MPC actually did.

In [ ]:
TARGET_DATE = "2026-07-23"
BAND = 0.05

target_row = next(r for r in combined if r["meeting_date"] == TARGET_DATE)
target_index = float(target_row["combined_index"])
print(f"Combined index for {TARGET_DATE}: {target_index:.4f}")

comparables = [
    (float(r["combined_index"]), r["meeting_date"], action_by_date.get(r["meeting_date"]))
    for r in combined
    if abs(float(r["combined_index"]) - target_index) <= BAND and r["meeting_date"] != TARGET_DATE
]
comparables.sort()

tally = Counter(a for _, _, a in comparables if a)
n = sum(tally.values())
print(f"\n{len(comparables)} historically comparable meetings (index within {BAND} of {target_index:.3f}), "
      f"{n} with a known rate decision (1 predates the official rate series' 2002 coverage):")
for action, count in tally.most_common():
    print(f"  {action:5s}  {count:2d}  ({count/n:.0%})")

# same check, narrowed to meetings that (like July 2026) immediately followed a hike
dates_sorted = sorted(r["meeting_date"] for r in statements if r.get("meeting_date"))
def prev_action(date):
    i = dates_sorted.index(date)
    return action_by_date.get(dates_sorted[i - 1]) if i > 0 else None

post_hike_comparables = [(v, d, a) for v, d, a in comparables if prev_action(d) == "hike"]
tally2 = Counter(a for _, _, a in post_hike_comparables if a)
n2 = sum(tally2.values())
print(f"\nNarrowed to the {len(post_hike_comparables)} that also followed a hike (July 2026's own position in the cycle):")
for action, count in tally2.most_common():
    print(f"  {action:5s}  {count:2d}  ({count/n2:.0%})")

## 9. Appendix: statements vs. speeches vs. the decision itself

One more check, scoring the two corpora entirely separately (no pooling this time): from
the start of inflation targeting (February 2000) onward, does a statement's own tone track
the decision it announces better or worse than the tone of speeches given in the weeks
*before* that meeting? `build_speech_statement_correlation.py` correlates each corpus's
index against the actual rate move (in basis points) and breaks down the mean index by
decision, for statements and speeches independently.

In [ ]:
import build_speech_statement_correlation
build_speech_statement_correlation.main()

Statements correlate meaningfully with what they're about to announce; speeches, scored on
their own content in the window before a meeting, correlate far more weakly with the
decision that follows -- directionally right, but nowhere near as separated. See the
article's Appendix for the full discussion.

---

See `README.md` for the full repo map, known limitations, and the "Methodology & AI Usage" note.